In [1]:
import pandas as pd

In [3]:
import sys
from pathlib import Path

# Project root folder
PROJECT_ROOT = Path(r"D:\AI Projects\Credit-Risk-Knowledge-Assistant")

# Add project root to Python's module search path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)

Project root added: D:\AI Projects\Credit-Risk-Knowledge-Assistant


In [4]:
from app.services.retriever_service import (
    retrieve_documents
)

from app.services.prompt_service import (
    get_rag_prompt,
    format_context
)

from app.services.llm_service import (
    get_llm
)

In [6]:
##Test the complete flow manually

question = (
    "What objective is stated in Chapter 1 "
    "of Ind AS 109 Financial Instruments?"
)

In [7]:
documents = retrieve_documents(
    question=question,
    search_type="similarity",
    top_k=5
)

d:\AI Projects\Credit-Risk-Knowledge-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5176.60it/s]


In [8]:
print("Retrieved chunks:", len(documents))

Retrieved chunks: 5


In [9]:
## Format retrieved context

context = format_context(
    documents )

print(context[:2000])

Source: INDAS109.pdf
Page: 1
Content:
246 
 
 
Indian Accounting Standard (Ind AS) 109 
Financial Instruments 
 
(The Indian Accounting Standard includes paragraphs set in bold type and plain 
type, which have equal authority. Paragraphs in bold type indicate the main 
principles.) 
 
 
Chapter 1 Objective 
 
1.1 The objective of this Standard is to establish principles for the 
financial reporting of financial assets and financial liabilities that will 
present relevant and useful information to users of financial 
statements for their assessment of the amounts, timi ng and uncertainty 
of an entity’s future cash flows.  
 
Chapter 2 Scope 
 
2.1 This Standard shall be applied by all entities to all types of 
financial instruments except:  
 
 
(a) those interests in subsidiaries, associates and joint ventures 
that are accounted for in accordance with Ind AS1 10 
ConsolidatedFinancial Statements , I nd AS 27  Separate 
Financial Statements orInd AS 28 Investments in Associates

---



In [10]:
## Build prompt

prompt = get_rag_prompt()

messages = prompt.invoke(
    {
        "context": context,
        "question": question
    }
)

In [11]:
### Invoke LLM

llm = get_llm()

response = llm.invoke(
    messages
)

print(response.content)

The objective stated in Chapter 1 of Ind AS 109 is:

> “The objective of this Standard is to establish principles for the financial reporting of financial assets and financial liabilities that will present relevant and useful information to users of financial statements for their assessment of the amounts, timing and uncertainty of an entity’s future cash flows.”【INDAS109.pdf, page 1】


In [12]:
## Extract sources separately

def extract_sources(documents):

    sources = []

    seen_sources = set()

    for document in documents:

        source_file = document.metadata.get(
            "source_file",
            "Unknown"
        )

        page_number = document.metadata.get(
            "page_number",
            "Unknown"
        )

        chunk_id = document.metadata.get(
            "chunk_id"
        )

        source_key = (
            source_file,
            page_number
        )

        # Avoid duplicate source/page combinations
        if source_key not in seen_sources:

            sources.append(
                {
                    "source_file": source_file,
                    "page_number": page_number,
                    "chunk_id": chunk_id
                }
            )

            seen_sources.add(
                source_key
            )

    return sources

In [13]:
sources = extract_sources(
    documents
)

sources

[{'source_file': 'INDAS109.pdf', 'page_number': 1, 'chunk_id': 0},
 {'source_file': 'INDAS109.pdf', 'page_number': 188, 'chunk_id': 621},
 {'source_file': 'INDAS109.pdf', 'page_number': 128, 'chunk_id': 413},
 {'source_file': 'INDAS109.pdf', 'page_number': 48, 'chunk_id': 139},
 {'source_file': 'INDAS109.pdf', 'page_number': 55, 'chunk_id': 160}]

In [14]:
## Build notebook version of answer_question()


def answer_question(
    question,
    top_k=5,
    filter_metadata=None
):

    # ---------------------------------
    # Validate question
    # ---------------------------------

    if not question or not question.strip():
        raise ValueError(
            "Question cannot be empty."
        )

    # ---------------------------------
    # Retrieve context
    # ---------------------------------

    documents = retrieve_documents(
        question=question,
        search_type="similarity",
        top_k=top_k,
        filter_metadata=filter_metadata
    )

    # ---------------------------------
    # Format context
    # ---------------------------------

    context = format_context(
        documents
    )

    # ---------------------------------
    # Build prompt
    # ---------------------------------

    prompt = get_rag_prompt()

    messages = prompt.invoke(
        {
            "context": context,
            "question": question
        }
    )

    # ---------------------------------
    # Invoke LLM
    # ---------------------------------

    llm = get_llm()

    response = llm.invoke(
        messages
    )

    # ---------------------------------
    # Extract sources
    # ---------------------------------

    sources = extract_sources(
        documents
    )

    # ---------------------------------
    # Return structured result
    # ---------------------------------

    return {
        "question": question,
        "answer": response.content,
        "sources": sources,
        "retrieved_chunks": len(documents)
    }

In [15]:
result = answer_question(
    "What objective is stated in Chapter 1 "
    "of Ind AS 109 Financial Instruments?"
)

result

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8574.07it/s]


{'question': 'What objective is stated in Chapter 1 of Ind AS 109 Financial Instruments?',
 'answer': 'The objective stated in Chapter\u202f1 of Ind\u202fAS\u202f109 is:\n\n> “The objective of this Standard is to establish principles for the financial reporting of financial assets and financial liabilities that will present relevant and useful information to users of financial statements for their assessment of the amounts, timing and uncertainty of an entity’s future cash flows.”【INDAS109.pdf, page\u202f1】',
 'sources': [{'source_file': 'INDAS109.pdf', 'page_number': 1, 'chunk_id': 0},
  {'source_file': 'INDAS109.pdf', 'page_number': 188, 'chunk_id': 621},
  {'source_file': 'INDAS109.pdf', 'page_number': 128, 'chunk_id': 413},
  {'source_file': 'INDAS109.pdf', 'page_number': 48, 'chunk_id': 139},
  {'source_file': 'INDAS109.pdf', 'page_number': 55, 'chunk_id': 160}],
 'retrieved_chunks': 5}

In [16]:
## Inspect cleanly

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:

    print(
        f"- {source['source_file']} "
        f"| Page {source['page_number']}"
    )

QUESTION:
What objective is stated in Chapter 1 of Ind AS 109 Financial Instruments?

ANSWER:
The objective stated in Chapter 1 of Ind AS 109 is:

> “The objective of this Standard is to establish principles for the financial reporting of financial assets and financial liabilities that will present relevant and useful information to users of financial statements for their assessment of the amounts, timing and uncertainty of an entity’s future cash flows.”【INDAS109.pdf, page 1】

SOURCES:
- INDAS109.pdf | Page 1
- INDAS109.pdf | Page 188
- INDAS109.pdf | Page 128
- INDAS109.pdf | Page 48
- INDAS109.pdf | Page 55


In [17]:
## Test unsupported question

result = answer_question(
    "How do I make Italian pasta?"
)

print(result["answer"])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3732.01it/s]


I could not find enough information in the provided documents.


In [19]:
## Test actual project questions

test_questions = [
    (
        "What objective is stated in Chapter 1 "
        "of Ind AS 109 Financial Instruments?"
    ),

    "What is expected credit loss?",

    (
        "What is significant increase "
        "in credit risk?"
    ),

    (
        "What are the rules for income "
        "recognition on NPAs?"
    ),

    (
        "What guidelines apply to "
        "bank finance to NBFCs?"
    )
]

In [20]:
for question in test_questions:

    print("\n" + "=" * 100)

    result = answer_question(
        question
    )

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")

    for source in result["sources"]:

        print(
            f"{source['source_file']} "
            f"| Page {source['page_number']}"
        )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3787.26it/s]



QUESTION:
What objective is stated in Chapter 1 of Ind AS 109 Financial Instruments?

ANSWER:
The objective stated in Chapter 1 of Ind AS 109 is:

> “The objective of this Standard is to establish principles for the financial reporting of financial assets and financial liabilities that will present relevant and useful information to users of financial statements for their assessment of the amounts, timing and uncertainty of an entity’s future cash flows.”【INDAS109.pdf, page 1】

SOURCES:
INDAS109.pdf | Page 1
INDAS109.pdf | Page 188
INDAS109.pdf | Page 128
INDAS109.pdf | Page 48
INDAS109.pdf | Page 55



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2354.98it/s]



QUESTION:
What is expected credit loss?

ANSWER:
**Expected credit loss**  
A probability‑weighted estimate of the credit losses that are expected to occur over the expected life of a financial instrument. It is the present value of all cash shortfalls – the difference between the cash flows that are due under the contract and the cash flows that the entity expects to receive (including timing of payments)【INDAS109.pdf, page 120】.

SOURCES:
INDAS109.pdf | Page 120
INDAS109.pdf | Page 123
INDAS109.pdf | Page 125
INDAS109.pdf | Page 119
INDAS109.pdf | Page 50



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10451.77it/s]



QUESTION:
What is significant increase in credit risk?

ANSWER:
**Significant increase in credit risk**  
A credit risk is considered to have increased significantly since initial recognition when, on an individual or collective basis, the entity determines that the risk of default or loss has risen to a level that requires lifetime expected credit‑loss measurement rather than the 12‑month expected credit‑loss amount.  
The assessment must be based on **all reasonable and supportable information available without undue cost or effort**, including both historical and forward‑looking data.  Factors that may indicate such an increase include:  

* Changes in the borrower’s business or organisational structure that affect its ability to meet debt obligations.  
* Significant increases in credit risk on other financial instruments of the same borrower.  
* Actual or expected adverse changes in the regulatory, economic, or technological environment that reduce the borrower’s ability to pay.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12347.12it/s]



QUESTION:
What are the rules for income recognition on NPAs?

ANSWER:
**Rules for income recognition on NPAs (as per the Master Circular)**  

| Situation | Rule | Source |
|-----------|------|--------|
| **General** – any advance that turns NPA | No interest should be charged or taken to the income account. The interest that has already been credited must be reversed by debiting the Profit‑and‑Loss account, and further interest application must be stopped. | Master Circular – page 7 (3.1.1) & page 8 (3.4) |
| **Interest on certain secured advances** – advances against Term Deposits, National Savings Certificates (NSCs), Kisan Vikas Patras (KVPs) and life‑insurance policies | Interest may be taken to the income account on the due date, provided an adequate margin is available in the accounts. | Master Circular – page 7 (3.1.2) |
| **Fees/commissions from renegotiation or rescheduling** | Recognise on an accrual basis over the period covered by the renegotiated or rescheduled extension

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3493.53it/s]



QUESTION:
What guidelines apply to bank finance to NBFCs?

ANSWER:


SOURCES:
Master Circular - Bank Finance to Non-Banking Financial Companies (NBFCs).pdf | Page 3
Master Circular - Bank Finance to Non-Banking Financial Companies (NBFCs).pdf | Page 6
Master Circular - Bank Finance to Non-Banking Financial Companies (NBFCs).pdf | Page 5
Master Circular - Bank Finance to Non-Banking Financial Companies (NBFCs).pdf | Page 9
Master Circular - Bank Finance to Non-Banking Financial Companies (NBFCs).pdf | Page 7
